# Inférence avec le modèle fine-tuné (test local)

Remplacer `MODEL_URI` / `TEST_DATA_URI` par les URI GCS des artifacts `model` et `test_output_path` visibles dans la console Vertex Pipelines.

In [ ]:
import json
import os
import re
import sys
from typing import Any

import pandas as pd
import torch
from google.cloud import storage
from transformers import AutoModelForCausalLM, AutoTokenizer

sys.path.append("..")  # pour importer src.constants depuis notebooks/

DEVICE = "cuda:0" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

MODEL_URI = "gs://<bucket>/pipeline_root/.../fine-tuning-component/model"
TEST_DATA_URI = "../data/test_sustained.csv"  # ou "gs://<bucket>/pipeline_root/.../test_output"
LOCAL_MODEL_DIR = "../phi3-mini-sustained-adapter"

## 1. Fonctions utilitaires

In [ ]:
def download_model(model_uri: str, local_dir: str):
    """Download model from GCS to local directory."""
    bucket_name, prefix = model_uri.removeprefix("gs://").split("/", 1)
    client = storage.Client()
    for blob in client.list_blobs(bucket_name, prefix=prefix):
        if blob.name.endswith("/"):
            continue
        local_path = os.path.join(local_dir, os.path.relpath(blob.name, prefix))
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        blob.download_to_filename(local_path)
        print(f"Téléchargé : {local_path}")


def build_prompt(tokenizer: AutoTokenizer, instruction: str) -> str:
    """Build a prompt from the user's message applying the chat template."""
    messages = [{"role": "user", "content": instruction}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_response(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    **kwargs: Any,
) -> str:
    """Generate a response from the model given a prompt."""
    tokenized = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**tokenized, eos_token_id=tokenizer.eos_token_id, **kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=False)


def extract_response(model_output: str) -> str:
    """Extract the actual response from the model output."""
    match = re.search(r"<\|assistant\|>(.*?)(?:<\|end\|>|<\|endoftext\|>|$)", model_output, re.DOTALL)
    return match.group(1).strip() if match else model_output.strip()

In [ ]:
# Vérification de extract_response sur l'exemple du brief
extract_response(
    "<|user|>\nI run a small coffee shop. What are Scope 3 emissions? Explain for a beginner.<|end|>\n<|assistant|>\nDefinition: Scope 3 emissions are the indirect emissions in your value chain.\nExample: For your coffee shop, they include growing the coffee beans you buy.\nAction: List your five biggest purchases.\nCheck: Is purchased electricity Scope 2 or Scope 3?<|end|>"
)

## 2. Téléchargement et chargement du modèle

In [ ]:
download_model(MODEL_URI, LOCAL_MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
# transformers détecte adapter_config.json et charge le modèle de base + les adaptateurs LoRA
model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL_DIR, torch_dtype=torch.bfloat16, device_map=DEVICE)
model.eval()
print(build_prompt(tokenizer, "I run a small coffee shop. What are Scope 3 emissions? Explain for a beginner."))

## 3. Prédictions sur les 3 premières lignes du jeu de test

In [ ]:
test_df = pd.read_csv(TEST_DATA_URI, keep_default_na=False)
messages = test_df["messages"].apply(json.loads)

rows = []
for i, conversation in enumerate(messages[:3]):
    instruction, reference = conversation[0]["content"], conversation[1]["content"]
    output = generate_response(model, tokenizer, build_prompt(tokenizer, instruction), max_new_tokens=256, do_sample=False)
    rows.append({"user_input": instruction, "reference": reference, "extracted_response": extract_response(output),
                 "learner_level": test_df["learner_level"][i], "business_sector": test_df["business_sector"][i]})

predictions_df = pd.DataFrame(rows)
predictions_df.to_csv("../data/predictions_sustained.csv", index=False)
predictions_df